# MAVEable genes for the Element imaging marker panels

Pick 100 genes from `MAVEable_final_is_feasible.csv` whose variant effects are most likely to be
readable with the Element V2 protein subpanels (all five panels run in parallel):

| Panel | Markers |
|---|---|
| Expanded Morphology | Golgi, ER, α-Tubulin, Lysosome |
| Alternative Morphology | β-Actin, Plasma membrane, Lamin, Nucleus |
| Neuro Morphology | Nucleophosmin, βIII-Tubulin, GFAP, MAP2 |
| Cell Signaling | Phospho-ERK1, ERK1/2, Phospho-HSP27, Phospho-p38 |
| Immune Subtyping | CD8A, CD4, CD19 |

Genes are filled into the 100 slots by evidence tier, then ranked within a tier by ClinVar content:

1. **Direct / functional readout** – the gene encodes a marker antigen, or sits in the RAS/MAPK or p38
   pathway that the phospho-markers report on
2. **Marked compartment + validated mislocalization** – main HPA location is a marked compartment *and*
   Lacoste et al. 2025 found pathogenic variants that mislocalize between marked compartments
3. **Marked compartment** – main HPA location is a marked compartment

Filters: protein length < 592 aa (same cutoff as the Broad scan list) and ≥ 20 missense VUS.

In [1]:
import pandas as pd
import numpy as np

In [2]:
maveable = pd.read_csv('MAVEable_final_is_feasible.csv', low_memory=False)

# missense counts used for ranking; Pathogenic/Likely pathogenic and Benign/Likely benign are the
# calibration controls a MAVE needs alongside the VUS it is trying to resolve
maveable['VUS'] = maveable['Uncertain significance']
maveable['P_LP'] = maveable['Pathogenic'] + maveable['Likely pathogenic']
maveable['B_LB'] = maveable['Benign'] + maveable['Likely benign']

## 1. Marker definitions

In [3]:
# Genes that encode a marker antigen: the antibody stains the variant protein itself, so abundance
# and localization of each variant are read directly. Antibody targets for the generic organelle
# markers (Golgi, ER, lysosome, plasma membrane, nucleus) aren't specified on the slide, so only
# GM130 (GOLGA2) is assumed there; MAP2 and NPM1/CD4 are absent or filtered out below.
direct_markers = {
    'GOLGA2': 'Golgi (GM130)',
    'TUBA1A': 'α-Tubulin', 'TUBA4A': 'α-Tubulin',
    'ACTB': 'β-Actin',
    'LMNA': 'Lamin', 'LMNB1': 'Lamin', 'LMNB2': 'Lamin',
    'NPM1': 'Nucleophosmin',
    'TUBB3': 'βIII-Tubulin',
    'GFAP': 'GFAP',
    'MAPK1': 'ERK1/2',
    'HSPB1': 'Phospho-HSP27',
    'CD8A': 'CD8A', 'CD4': 'CD4', 'CD19': 'CD19',
}

# RAS/MAPK pathway (RASopathy genes and upstream RTKs): variant effects shift phospho-ERK
erk_pathway = ['HRAS', 'KRAS', 'NRAS', 'RIT1', 'RRAS2', 'MRAS', 'SOS1', 'SOS2', 'PTPN11', 'SHOC2',
               'PPP1CB', 'RAF1', 'BRAF', 'MAP2K1', 'MAP2K2', 'MAPK1', 'NF1', 'SPRED1', 'CBL', 'LZTR1',
               'SYNGAP1', 'RASA1', 'RASA2', 'SPRY1', 'RASGRP1', 'RPS6KA3',
               'EGFR', 'FGFR1', 'FGFR2', 'FGFR3', 'KIT', 'PDGFRA', 'RET', 'MET']
# p38 / HSP27 stress axis: variant effects shift phospho-p38 and phospho-HSP27
p38_pathway = ['MAP3K7', 'TAB2', 'MAP3K20', 'MAP3K1']

# Human Protein Atlas compartments that a marker labels. Nucleoplasm and Cytosol are left out:
# almost every gene has them, so they don't single out a marker-specific readout.
compartment_markers = {
    'Golgi apparatus': 'Golgi',
    'Endoplasmic reticulum': 'ER',
    'Microtubules': 'α-Tubulin', 'Mitotic spindle': 'α-Tubulin', 'Cytokinetic bridge': 'α-Tubulin',
    'Lysosomes': 'Lysosome', 'Endosomes': 'Lysosome',
    'Actin filaments': 'β-Actin', 'Focal adhesion sites': 'β-Actin',
    'Plasma membrane': 'Plasma membrane', 'Cell Junctions': 'Plasma membrane',
    'Nuclear membrane': 'Lamin',
    'Nucleoli': 'Nucleophosmin', 'Nucleoli fibrillar center': 'Nucleophosmin', 'Nucleoli rim': 'Nucleophosmin',
    'Intermediate filaments': 'GFAP',
}

# Compartment names used in the Lacoste 'Mislocalization phenotype' column (e.g. 'Plasma membrane>ER')
# mapped to the marker that would see the move. 'Cytoplasm', 'Foci' and 'Mitochondria' have no marker.
lacoste_markers = {
    'plasma membrane': 'Plasma membrane', 'cellular periphery': 'Plasma membrane',
    'er': 'ER', 'golgi apparatus': 'Golgi', 'golgi': 'Golgi',
    'tubulin': 'α-Tubulin', 'actin': 'β-Actin', 'intermediate filaments': 'GFAP',
    'nucleus': 'Nucleus', 'nucleolus': 'Nucleophosmin',
    'nuclear membrane': 'Lamin', 'nuclear periphery': 'Lamin', 'lysosome': 'Lysosome',
}

## 2. Annotate each gene with the markers that can read it out

In [4]:
def main_location_markers(main_location):
    if not isinstance(main_location, str):
        return []
    return sorted({compartment_markers[loc] for loc in main_location.split(';') if loc in compartment_markers})

def mislocalization_markers(phenotype):
    # phenotypes look like 'Plasma membrane>ER; Golgi apparatus>ER'
    if not isinstance(phenotype, str):
        return []
    parts = [p.strip().lower() for ph in phenotype.split('; ') for p in ph.replace('>', ',').split(',')]
    return sorted({lacoste_markers[p] for p in parts if p in lacoste_markers})

maveable['direct_marker'] = maveable['GeneSymbol'].map(direct_markers)
maveable['pathway'] = np.select(
    [maveable['GeneSymbol'].isin(erk_pathway), maveable['GeneSymbol'].isin(p38_pathway)],
    ['RAS/MAPK (pERK, ERK1/2)', 'p38 (p-p38, pHSP27)'], default=None)
maveable['compartment_markers'] = maveable['Main location'].apply(main_location_markers)
# Mislocalization phenotype is only filled for genes with Mislocalized == 1
maveable['mislocalization_markers'] = maveable['Mislocalization phenotype'].apply(mislocalization_markers)

has_compartment = maveable['compartment_markers'].str.len() > 0
has_mislocalization = maveable['mislocalization_markers'].str.len() > 0
maveable['tier'] = np.select(
    [maveable['direct_marker'].notna() | maveable['pathway'].notna(),
     has_compartment & has_mislocalization,
     has_compartment],
    [1, 2, 3], default=0)

def readout_markers(row):
    markers = set(row['compartment_markers']) | set(row['mislocalization_markers'])
    if isinstance(row['direct_marker'], str):
        markers.add(row['direct_marker'])
    if isinstance(row['pathway'], str):
        markers.add(row['pathway'])
    return '; '.join(sorted(markers))

maveable['readout_markers'] = maveable.apply(readout_markers, axis=1)
maveable['tier'].value_counts().sort_index()

tier
0    3570
1      52
2      38
3    1154
Name: count, dtype: int64

## 3. Filter and rank

In [5]:
# Length < 592 aa so the whole coding sequence can be scanned (same cutoff as the Broad scan list);
# >= 20 VUS so there is something for the assay to resolve
candidates = maveable[(maveable['tier'] > 0) & (maveable['Length'] < 592) & (maveable['VUS'] >= 20)].copy()

dropped_direct = maveable[(maveable['tier'] == 1) & ~maveable['GeneSymbol'].isin(candidates['GeneSymbol'])]
print('Direct/pathway genes excluded by length or VUS filters:')
print(dropped_direct[['GeneSymbol', 'Length', 'VUS']].sort_values('Length').to_string(index=False))

# Within a tier: more VUS is better (log scale so huge counts don't swamp everything), plus a bonus
# for having pathogenic and benign controls to calibrate the assay
candidates['clinvar_score'] = (np.log10(candidates['VUS'] + 1)
                               + 0.5 * (candidates['P_LP'] >= 3)
                               + 0.5 * (candidates['B_LB'] >= 3))

candidates = candidates.sort_values(['tier', 'clinvar_score', 'VUS'], ascending=[True, False, False])
top100 = candidates.head(100).copy()
top100.insert(0, 'rank', range(1, len(top100) + 1))
top100['tier'].value_counts().sort_index()

Direct/pathway genes excluded by length or VUS filters:
GeneSymbol  Length  VUS
      NPM1   294.0    6
       CD4   458.0   18
    PTPN11   593.0  439
    MAP3K7   606.0  146
     LMNB2   620.0  287
      RAF1   648.0  506
      LMNA   664.0  754
      TAB2   693.0  196
   RPS6KA3   740.0  124
      BRAF   766.0  426
   RASGRP1   797.0  188
   MAP3K20   800.0  132
     FGFR3   806.0  482
     FGFR2   821.0  295
     FGFR1   822.0  384
     LZTR1   840.0 1966
     RASA2   850.0  407
       CBL   906.0 1192
       KIT   976.0 1438
    GOLGA2  1002.0  150
     RASA1  1047.0  594
    PDGFRA  1089.0 1724
       RET  1114.0 1829
      EGFR  1210.0 1649
      SOS2  1332.0  810
      SOS1  1333.0  877
   SYNGAP1  1343.0  522
       MET  1390.0 2148
    MAP3K1  1512.0  220
       NF1  2839.0 5026


tier
1    22
2    25
3    53
Name: count, dtype: int64

## 4. Coverage per marker

In [6]:
marker_counts = top100['readout_markers'].str.split('; ').explode().value_counts()
marker_counts

readout_markers
Plasma membrane            40
Golgi                      23
ER                         22
RAS/MAPK (pERK, ERK1/2)    13
Nucleophosmin              10
Lamin                       7
GFAP                        6
α-Tubulin                   6
β-Actin                     3
Nucleus                     3
Phospho-HSP27               1
βIII-Tubulin                1
CD19                        1
ERK1/2                      1
CD8A                        1
Name: count, dtype: int64

In [7]:
output_columns = ['rank', 'GeneSymbol', 'tier', 'readout_markers', 'direct_marker', 'pathway',
                  'compartment_markers', 'mislocalization_markers', 'Mislocalization phenotype',
                  'VUS', 'P_LP', 'B_LB', 'clinvar_score', 'Length', 'Main location', 'Additional location',
                  'class', 'location_feasible', 'HAP1', 'T_cell', 'B_cell', 'Protein names', 'PhenotypeList']
element_top100 = top100[output_columns].copy()
for col in ['compartment_markers', 'mislocalization_markers']:
    element_top100[col] = element_top100[col].str.join('; ')
element_top100.to_csv('MAVEable_element_marker_top100_20261006.csv', index=False)
element_top100[['rank', 'GeneSymbol', 'tier', 'readout_markers', 'VUS', 'P_LP', 'B_LB', 'Length']]

,rank,GeneSymbol,tier,readout_markers,VUS,P_LP,B_LB,Length
4075,1,SPRED1,1,"RAS/MAPK (pERK, ERK1/2)",538,7,15,444.0
2429,2,MAP2K2,1,"RAS/MAPK (pERK, ERK1/2)",357,19,10,400.0
3812,3,SHOC2,1,"RAS/MAPK (pERK, ERK1/2)",307,4,6,582.0
1912,4,HRAS,1,"RAS/MAPK (pERK, ERK1/2)",222,27,9,189.0
1627,5,GFAP,1,GFAP,191,58,15,432.0
...,...,...,...,...,...,...,...,...
1478,96,FKBP10,3,Golgi,185,8,3,582.0
1408,97,FAS,3,Plasma membrane,182,30,6,335.0
2454,98,MATN3,3,ER; Golgi,179,7,11,486.0
173,99,ALG6,3,ER,178,7,9,507.0


## 5. Neuro focus: 100 genes with neurological disease associations

Same marker readouts, tiers, length/VUS filters and ranking as above, restricted to genes with a
neurological disease in GenCC (Definitive/Strong/Moderate, the same confidence cutoff as the main
pipeline) or on the neurodegeneration gene list used in `20260911_for_CZI.ipynb`.

In [8]:
gencc = pd.read_csv('gencc-submissions_20260911.csv', low_memory=False)
gencc_moderate = gencc[gencc['classification_title'].isin(['Definitive', 'Strong', 'Moderate'])]

# Neurological disease terms. Hearing loss and retinal disease are left out on purpose;
# 'ciliary dyskinesia' and 'adrenocortical' are guarded against by the patterns below.
neuro_pattern = (
    r'epilep|seizure|encephalopath|intellectual|neurodevelopment|developmental delay|autis|ataxia|'
    r'spastic paraplegia|charcot-marie-tooth|neuropathy|neuronopathy|parkinson|alzheimer|amyotrophic|'
    r'frontotemporal|dementia|dystonia|leukodystroph|leukoencephalopath|hypomyelinat|white matter|'
    r'microcephal|lissenceph|polymicrogyria|pachygyria|\bcortical|cerebell|neurodegenerat|huntington|'
    r'chorea|motor neuron|spinal muscular atrophy|alexander disease|neuronal ceroid|brain|hydroceph|'
    r'holoprosenceph|schizophrenia|migraine|myoclon|(?<!ciliary )dyskinesia|joubert|rett|angelman|'
    r'tubulinopathy|hereditary sensory|neurolog|neuromuscular|paroxysmal'
)
neuro_gencc = gencc_moderate[gencc_moderate['disease_title'].str.contains(neuro_pattern, case=False, na=False)]
neuro_diseases = neuro_gencc.groupby('gene_symbol')['disease_title'].apply(lambda x: '; '.join(sorted(set(x))))

# Neurodegeneration genes from 20260911_for_CZI.ipynb
neurodegeneration_genes = [
    'APP', 'PSEN1', 'PSEN2', 'SORL1', 'MAPT', 'ADAM10', 'LRRK2', 'GBA1', 'TMEM175', 'VPS13C', 'ATP13A2',
    'RAB32', 'SNCA', 'PRKN', 'PINK1', 'PARK7', 'VPS35', 'SYNJ1', 'DNAJC6', 'CHCHD2', 'SH3GL2', 'RIT2',
    'SV2C', 'TARDBP', 'C9orf72', 'FUS', 'SOD1', 'GRN', 'SQSTM1', 'OPTN', 'VCP', 'UBQLN2', 'CHCHD10',
    'ANXA11', 'TBK1', 'UNC13A', 'ATXN2', 'KIF5A', 'PFN1', 'TUBA4A', 'NEFH', 'DCTN1', 'SETX', 'MATR3',
    'HNRNPA1', 'HNRNPA2B1',
]

candidates['neuro_disease'] = candidates['GeneSymbol'].map(neuro_diseases)
candidates['neurodegeneration_list'] = candidates['GeneSymbol'].isin(neurodegeneration_genes)
neuro_candidates = candidates[candidates['neuro_disease'].notna() | candidates['neurodegeneration_list']]
print(f'Neuro candidates passing marker, length and VUS filters: {len(neuro_candidates)}')
neuro_candidates['tier'].value_counts().sort_index()

Neuro candidates passing marker, length and VUS filters: 178


tier
1      8
2      8
3    162
Name: count, dtype: int64

In [9]:
# candidates is already sorted by tier, then clinvar_score
neuro_top100 = neuro_candidates.head(100).copy()
neuro_top100.insert(0, 'rank', range(1, len(neuro_top100) + 1))

neuro_output_columns = output_columns[:4] + ['neuro_disease', 'neurodegeneration_list'] + output_columns[4:]
element_neuro_top100 = neuro_top100[neuro_output_columns].copy()
for col in ['compartment_markers', 'mislocalization_markers']:
    element_neuro_top100[col] = element_neuro_top100[col].str.join('; ')
element_neuro_top100.to_csv('MAVEable_element_neuro_top100_20261006.csv', index=False)

print(element_neuro_top100['tier'].value_counts().sort_index())
print(element_neuro_top100['readout_markers'].str.split('; ').explode().value_counts())
element_neuro_top100[['rank', 'GeneSymbol', 'tier', 'readout_markers', 'VUS', 'P_LP', 'B_LB', 'Length', 'neuro_disease']]

tier
1     8
2     8
3    84
Name: count, dtype: int64
readout_markers
Plasma membrane            41
Golgi                      24
Nucleophosmin              13
ER                         11
α-Tubulin                  10
Lamin                       7
β-Actin                     4
GFAP                        3
Phospho-HSP27               1
βIII-Tubulin                1
ERK1/2                      1
RAS/MAPK (pERK, ERK1/2)     1
Name: count, dtype: int64


,rank,GeneSymbol,tier,readout_markers,VUS,P_LP,B_LB,Length,neuro_disease
1627,1,GFAP,1,GFAP,191,58,15,432.0,Alexander disease
1928,2,HSPB1,1,Phospho-HSP27; Plasma membrane,171,20,3,205.0,Charcot-Marie-Tooth disease axonal type 2F; ne...
2344,3,LMNB1,1,Lamin,147,4,13,586.0,adult-onset autosomal dominant demyelinating l...
4528,4,TUBB3,1,α-Tubulin; βIII-Tubulin,137,47,10,450.0,TUBB3-related tubulinopathy; complex cortical ...
56,5,ACTB,1,β-Actin,125,95,0,375.0,developmental malformations-deafness-dystonia ...
...,...,...,...,...,...,...,...,...,...
3181,96,PLD3,3,ER,113,0,10,490.0,leukodystrophy; spinocerebellar ataxia 46
4415,97,TOR1A,3,Lamin,106,1,7,332.0,early-onset generalized limb-onset dystonia
2237,98,KPTN,3,Plasma membrane,99,1,5,436.0,complex neurodevelopmental disorder; macroceph...
1293,99,ENTPD1,3,α-Tubulin,96,0,3,510.0,complex hereditary spastic paraplegia; heredit...
